# Supplementary Figure S1 — behavioural transects

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from metpy.interpolate import interpolate_to_points

from Gbim_import_dat import get_Gbim_data_chirpfield, get_Gbim_data_DC
import plotting as p

## Load behavioral responses

In [ ]:
[CDUR, CPAU, CPER, DCs, NPUL, rXY]= get_Gbim_data_chirpfield()
[CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]=get_Gbim_data_DC()

keys = ['CDUR', 'CPAU', 'CPER', 'CDC', 'NPUL', 'rXY']
data_cpf = dict(zip(keys, [CDUR, CPAU, CPER, DCs, NPUL, rXY]))
data_dc = dict(zip(keys, [CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]))

data_all = {key: np.concatenate((data_cpf[key], data_dc[key])) for key in keys}

## Interpolate the field

In [ ]:
max_len = 1000
step = 5

duri0 = paui0 = np.arange(0, max_len + 0, step)
duri, paui = np.meshgrid(duri0,paui0)

CDURinterp=np.concatenate((CDUR,[0, 0]))
CPAUinterp=np.concatenate((CPAU,[0, 1000]))
rXYinterp=np.concatenate((rXY,[0, 0]))

points = np.array([CDURinterp + np.random.randn(*CDURinterp.shape)/1_000, CPAUinterp+ np.random.randn(*CPAUinterp.shape)/1_000]).T
new_points = np.array([duri, paui]).T.reshape((-1, 2))

cpf = interpolate_to_points(points, np.clip(rXYinterp, 0, np.inf), new_points, interp_type='natural_neighbor', minimum_neighbors=4, rbf_smooth=0)
cpf = np.maximum(cpf, 0)
cpf = cpf.reshape((len(duri), len(paui))).T

## Show the four transect windows

In [ ]:
plt.figure(figsize=(12, 3))
plt.subplot(141)
plt.pcolormesh(duri0/1000, paui0/1000, cpf / np.nanmax(cpf), cmap='Greys', rasterized=True)
plt.axis('equal')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.axhspan(.160, .240, alpha=0.5)
plt.ylabel('Chirp pause [s]')
plt.xlabel('Chirp duration [s]')
plt.xticks([0,1])
plt.yticks([0,1])
plt.title("Duration")

plt.subplot(142)
plt.pcolormesh(duri0/1000, paui0/1000, cpf / np.nanmax(cpf), cmap='Greys', rasterized=True)
plt.axis('equal')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.axvspan(.120, .200, alpha=0.5)
plt.xticks([0,1])
plt.yticks([0,1])
plt.title("Pause")

plt.subplot(143)
plt.pcolormesh(duri0/1000, paui0/1000, cpf / np.nanmax(cpf), cmap='Greys', rasterized=True)
plt.axis('equal')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.fill([0, 0, .320, .240], [.240, .320, 0, 0], color='#1f77b4', alpha=.5)
plt.xticks([0,1])
plt.yticks([0,1])
plt.title("Duty cycle")

plt.subplot(144)
plt.pcolormesh(duri0/1000, paui0/1000, cpf / np.nanmax(cpf), cmap='Greys', rasterized=True)
plt.axis('equal')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.fill([0, 2/3, 1, 1], [0, 1, 1, 2/3], color='#1f77b4', alpha=.5)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xticks([0,1])
plt.yticks([0,1])
plt.title("Period")
p.label_axes(size=14, loc=(-.2, 1.02))
plt.savefig(f'{FIGURES}/fig_S1.pdf', bbox_inches='tight')